In [1]:
!pip install tensorflow gradio -q

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gradio as gr
print("Ready!")

Ready!


In [2]:
text = """
the radio had been silent for eleven years maya kept it on her desk anyway
on a tuesday in november the radio crackled and a faint voice came through
it was her father he said maya i found something they do not want found
she walked slowly to the old wooden desk and opened the bottom drawer
inside was a blue envelope with her name written in her father's handwriting
she sat on the cold floor and felt hope rise inside her chest for the first time
"""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

SEQ_LENGTH = 6
tokens = tokenizer.texts_to_sequences([text])[0]
sequences = [tokens[i-SEQ_LENGTH:i+1] for i in range(SEQ_LENGTH, len(tokens))]
sequences = np.array(pad_sequences(sequences, maxlen=SEQ_LENGTH+1, padding='pre'))

X, y = sequences[:,:-1], sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
print(f"Vocab: {total_words} | Sequences: {len(sequences)}")

Vocab: 63 | Sequences: 81


In [3]:
model = Sequential([
    Embedding(total_words, 32, input_length=SEQ_LENGTH),
    LSTM(64),
    Dense(total_words, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.0123 - loss: 4.1431
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0864 - loss: 4.1381
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0864 - loss: 4.1338
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0988 - loss: 4.1297
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0988 - loss: 4.1251
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0988 - loss: 4.1203
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0864 - loss: 4.1146
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0864 - loss: 4.1081
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0864 - loss: 4.1011
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0988 - loss: 4.0919
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0988 - loss: 4.0817
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1111 - lo

In [4]:
def generate(seed, words=30, temp=0.7):
    out = seed
    for _ in range(words):
        seq = pad_sequences([tokenizer.texts_to_sequences([out])[0]], maxlen=SEQ_LENGTH, padding='pre')
        probs = model.predict(seq, verbose=0)[0]
        probs = np.exp(np.log(probs+1e-10)/temp) / np.sum(np.exp(np.log(probs+1e-10)/temp))
        idx = np.random.choice(len(probs), p=probs)
        out += " " + [w for w,i in tokenizer.word_index.items() if i==idx][0]
    return out

gr.Interface(
    fn=lambda seed, words, temp: generate(seed, int(words), float(temp)),
    inputs=[
        gr.Textbox(label="Start your story", placeholder="the radio had been silent"),
        gr.Slider(10, 80, value=30, step=10, label="Words to generate"),
        gr.Slider(0.1, 1.0, value=0.7, step=0.1, label="Temperature")
    ],
    outputs=gr.Textbox(label="Generated Story", lines=6),
    title="LSTM Story Generator"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebeb46596c3ed31fb3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
